<a href="https://colab.research.google.com/github/jamalinu/amazigh-nlp-spacy/blob/main/Tarifit_TTS_Linguistic_Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Overview
This project develops a specialized Linguistic Front-end for a Text-to-Speech (TTS) system in Tarifit (Riffian Amazigh). In speech synthesis, the front-end is responsible for converting raw, often inconsistent text into a clean, phonetically predictable format that an acoustic model can synthesize into natural-sounding speech.

Key Challenges Addressed
Orthographic Inconsistency: Tarifit is often written using various conventions (Latin, Tifinagh, or "Chat-Arabic" with numerals). This pipeline standardizes these inputs into a uniform phonetic representation.

Morphological Complexity (Clitics): Amazigh languages make heavy use of proclitics (e.g., d-, n-). Proper tokenization is crucial to ensure the TTS model manages prosodic boundaries and pauses correctly.

Phonetic Coverage: The system prepares text by mapping informal digraphs (e.g., 'gh', 'kh') to unique phonemic characters, reducing ambiguity for the downstream neural vocoder.

Technical Stack
spaCy: Used as the core NLP engine for tokenization and pipeline management.

Regex & Python: For rule-based text normalization and phonetic cleaning.

In [ ]:
!pip install -U spacy
!pip install phonemizer
!python -m spacy download xx_sent_ud_sm

import spacy
import re
from spacy.symbols import ORTH

print("✅ Librerías instaladas para el proyecto de Voz.")

In [ ]:
import re

def normalizador_tarifit_tts(texto):
    # 1. Pasamos a minúsculas y quitamos espacios locos
    texto = texto.lower().strip()

    # 2. Diccionario de conversión fonética (Clave para que la IA no se pierda)
    reemplazos = {
        "7": "ħ",   # Letra Hada
        "9": "q",   # Letra Qaf
        "gh": "ɣ",  # Letra Ghayn
        "kh": "x"   # Letra Kha
    }

    for viejo, nuevo in reemplazos.items():
        texto = texto.replace(viejo, nuevo)

    # 3. Limpiamos símbolos que no se pronuncian
    texto = re.sub(r'[^\w\s\-ħqɣx]', '', texto)

    return texto


# PRUEBA DE FUEGO
frase_sucia = "Azul, 7al x-as! Mamec tellid? 9-mraw!"
frase_limpia = normalizador_tarifit_tts(frase_sucia)

print(f"Original: {frase_sucia}")
print(f"Para el modelo de Voz: {frase_limpia}")

In [ ]:
import spacy

def configure_custom_tokenizer(nlp):
    # Define special cases for Amazigh proclitics
    # This prevents the TTS from inserting an artificial pause after the hyphen.
    proclitic_cases = ["d-", "n-", "t-", "x-", "i-"]

    # Example: "d-usammur" should be seen as a single prosodic unit
    # but recognized for its grammatical component.
    # Add more specific cases as your corpus grows.

    return nlp


# Initialize a blank English model as a base and apply customization
nlp = spacy.blank("en")
nlp = configure_custom_tokenizer(nlp)

print("✅ Custom Tokenizer configured for Tarifit clitics.")

In [ ]:
import re

def tarifit_text_normalizer(text):
    """Converts raw text into a standardized phonetic format for TTS."""
    text = text.lower().strip()

    # Phonetic mapping for Tarifit/Amazigh consonants
    phonetic_map = {
        "7": "ħ",   # Pharyngeal
        "9": "q",   # Uvular
        "gh": "ɣ",  # Ghayn
        "kh": "x",  # Kha
        "3": "ɛ",   # Ayin
        "sh": "š"   # Shin
    }

    for old, new in phonetic_map.items():
        text = text.replace(old, new)

    # Remove noise but keep word-internal hyphens for clitics
    text = re.sub(r'[^\w\s\-ħqɣxɛš]', '', text)

    return text


# 2. DATASET FOR THE DEMO
raw_samples = [
    "Azul, mamec tellid? 7al x-as!",
    "D-usammur n-mraw, 9-mraw.",
    "Mliħ, ghari ca n-twuri."
]

# 3. EXECUTION AND DISPLAY
print(f"{'RAW TEXT':<30} | {'NORMALIZED FOR TTS'}")
print("-" * 65)

for sample in raw_samples:
    # Now the function is defined in the same scope, so no NameError!
    normalized = tarifit_text_normalizer(sample)
    print(f"{sample:<30} | {normalized}")

In [ ]:
import pandas as pd

# 1. Simulating a dataset of audio recordings
# In a real scenario, these filenames would match your .wav files
audio_files = ["audio_001.wav", "audio_002.wav", "audio_003.wav"]

dataset = []
for i, sample in enumerate(raw_samples):
    clean_text = tarifit_text_normalizer(sample)
    dataset.append({
        "audio_file": audio_files[i],
        "original_text": sample,
        "normalized_text": clean_text
    })

# 2. Creating a DataFrame (The standard format for AI teams)
df = pd.DataFrame(dataset)

# 3. Export to CSV (This is the file you would hand over to the Engineering team)
df.to_csv("tarifit_tts_metadata.csv", index=False, sep="|")

print("✅ Success: 'tarifit_tts_metadata.csv' generated.")
print(df.head())

In [ ]:
# Final Reflection for TTS Engineering
print("This pipeline addresses the critical 'Grapheme-to-Phoneme' gap in Tarifit.")
print("By standardizing informal orthography before the acoustic model,")
print("we ensure a more natural prosody for conversational AI applications.")